In [1]:
import os
import psycopg2
import keyring

# Define database connection details
DB_NAME = "image_dataset"  # ✅ Set DB_NAME before using it
DB_USER = "postgres"
DB_PASSWORD = keyring.get_password("PostgreSQL", "postgres")
DB_HOST = "localhost"
DB_PORT = "5432"

# Establish connection
try:
    conn = psycopg2.connect(
        dbname=DB_NAME,
        user=DB_USER,
        password=DB_PASSWORD,
        host=DB_HOST,
        port=DB_PORT
    )
    cursor = conn.cursor()
    print("✅ PostgreSQL connection established!")

except Exception as e:
    print(f"❌ Error connecting to database: {e}")


✅ PostgreSQL connection established!


In [2]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
import os
import time
import urllib.request

# Path to ChromeDriver
driver_path = r"C:\Program Files\chromedriver-win64\chromedriver.exe"

# Base directory for saving images
base_dir = r"C:\Users\postgres\yolo_project\mined"

# Initialize Selenium WebDriver correctly
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# Test WebDriver
driver.get("https://www.duckduckgo.com")
print("✅ ChromeDriver is working!")
driver.quit()


✅ ChromeDriver is working!


In [5]:
from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO

# Function to fetch images using DuckDuckGo API
def fetch_image(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=1)  # Get 1 image per category

    if results:
        image_url = results[0]["image"]
        print(f"🔍 {category}: {image_url}")

        # Fetch and display the image without saving
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        img.show()

    else:
        print(f"⚠️ No images found for {category}")

# Test with a refined query
fetch_image("construction crane")


🔍 construction crane: https://images.pexels.com/photos/1078853/pexels-photo-1078853.jpeg?cs=srgb&dl=construction-crane-1078853.jpg&fm=jpg


In [ ]:
# Loading 10 cranes

from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO

# Define categories
categories = ["construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
              "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest", 
              "safety goggles", "gloves", "boots", "harness", "respirator mask",
              "scaffolding", "barricade", "traffic cone", "construction sign", 
              "wheelbarrow", "ladder", "cables wiring", "unstable structure", 
              "exposed wiring", "falling debris", "fire risk", "oil spill"]

# Function to scrape images for a given category
def scrape_images(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=10)  # Retrieve 10 images
    image_urls = [img["image"] for img in results if "image" in img]

    if image_urls:
        print(f"🔍 {category}: {len(image_urls)} images found")
    else:
        print(f"⚠️ No images found for {category}")

    return image_urls

# Test the function with one category
scrape_images("construction crane")


🔍 construction crane: 10 images found


['https://images.pexels.com/photos/1078853/pexels-photo-1078853.jpeg?cs=srgb&dl=construction-crane-1078853.jpg&fm=jpg',
 'https://www.liebherr.com/shared/media/construction-machinery/tower-cranes/landingpage-tower-cranes/slider-products-solutions/lbc-slider1-turmdrehkrane.jpg',
 'https://www.mazzellacompanies.com/wp-content/uploads/2023/04/Tall-Tower-Crane-with-Clouds-Behind.jpg',
 'https://civilseek.com/wp-content/uploads/2017/07/tower-crane.jpg',
 'https://everythingcranes.com/wp-content/uploads/2021/11/types-of-construction-cranes.jpeg',
 'https://jlrnrwxhjoki5q.leadongcdn.com/cloud/jpBopKrmSRnpnrori/10416317_0.jpg',
 'https://heavyequipmentcollege.edu/wp-content/uploads/2022/06/A-Guide-to-Mobile-Cranes-in-the-Construction-Industry-Heavy-Equipment-Colleges-of-america-scaled.jpg',
 'https://rmscranes.com/wp-content/uploads/2022/06/RMS-Tower-Cranes-Accross-the-Skyline.jpg',
 'https://www.publicdomainpictures.net/pictures/110000/velka/construction-crane-1415658191KzZ.jpg',
 'https://im

In [9]:
from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO
import concurrent.futures

# Define categories
categories = ["construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
              "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
              "safety goggles", "gloves", "boots", "harness", "respirator mask",
              "scaffolding", "barricade", "traffic cone", "construction sign",
              "wheelbarrow", "ladder", "cables wiring", "unstable structure",
              "exposed wiring", "falling debris", "fire risk", "oil spill"]

# Function to fetch images using DuckDuckGo API
def fetch_image(category):
    ddg = DDGS()
    results = ddg.images(category, max_results=1)  # Retrieve only 1 image per category
    image_url = results[0]["image"] if results else None
    return category, image_url

# Function to download and display images
def display_image(category, image_url):
    if image_url:
        response = requests.get(image_url)
        img = Image.open(BytesIO(response.content))
        img.show()
        print(f"✅ {category}: {image_url}")
    else:
        print(f"⚠️ No image found for {category}")

# Run parallel execution using ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor() as executor:
    future_to_category = {executor.submit(fetch_image, category): category for category in categories}
    for future in concurrent.futures.as_completed(future_to_category):
        category = future_to_category[future]
        try:
            category, image_url = future.result()
            display_image(category, image_url)
        except Exception as exc:
            print(f"⚠️ {category} generated an exception: {exc}")

print("✅ Completed parallel image mining and display!")


def validate_url(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        return response.headers.get("content-type", "").startswith("image/")
    except requests.RequestException:
        return False

def fetch_image_safe(url):
    response = requests.get(url, timeout=10)
    if "image" in response.headers.get("content-type", ""):
        return Image.open(BytesIO(response.content))
    else:
        print(f"⚠️ Invalid image response for {url}")
        return None

try:
    img = Image.open(BytesIO(response.content))
    img.verify()  # Detect corruption without displaying
    img.show()
except Exception as e:
    print(f"❌ Error displaying image: {e}")

import traceback

def display_image(category, image_url):
    try:
        response = requests.get(image_url, timeout=10)
        img = Image.open(BytesIO(response.content))
        img.show()
        print(f"✅ {category}: {image_url}")

    except Exception as e:
        print(f"⚠️ Error loading {category}: {image_url}")
        print(traceback.format_exc())  # Shows full error details


⚠️ respirator mask generated an exception: https://duckduckgo.com/?q=respirator+mask 202 Ratelimit
⚠️ dump truck generated an exception: https://duckduckgo.com/?q=dump+truck 202 Ratelimit
⚠️ paver generated an exception: https://duckduckgo.com/?q=paver 202 Ratelimit
⚠️ unstable structure generated an exception: cannot identify image file <_io.BytesIO object at 0x000001770965D5D0>
✅ boots: http://image.sportsmansguide.com/adimgs/l/1/191004_ts.jpg
✅ safety vest: https://images-na.ssl-images-amazon.com/images/I/71zVQtogaaL._AC_UL1280_.jpg
✅ gloves: https://cdnimg.webstaurantstore.com/images/products/xxl/399963/1477981.jpg
✅ safety goggles: https://cdn11.bigcommerce.com/s-10f42/images/stencil/2560w/products/2273/5141/Sebring_Safety_Glasses_-_Clear_Tint__20669.1591791783.png?c=2
✅ cables wiring: https://www.thespruce.com/thmb/WXGFq-sr4TsErJNRc49HYBfSeyo=/5763x3842/filters:no_upscale():max_bytes(150000):strip_icc()/electrical-wiring-1152909_03_color_coding-49e8a933548d44c488495999ed836093.jp

In [ ]:
from duckduckgo_search import DDGS
import requests
from PIL import Image
from io import BytesIO
import concurrent.futures
import traceback

# Define categories
categories = ["construction crane", "bulldozer", "forklift", "excavator", "cement mixer",
              "dump truck", "backhoe", "loader", "paver", "hard hat", "safety vest",
              "safety goggles", "gloves", "boots", "harness", "respirator mask",
              "scaffolding", "barricade", "traffic cone", "construction sign",
              "wheelbarrow", "ladder", "cables wiring", "unstable structure",
              "exposed wiring", "falling debris", "fire risk", "oil spill"]

# Function to validate if a URL points to an actual image
def validate_url(url):
    try:
        response = requests.head(url, allow_redirects=True, timeout=5)
        return response.headers.get("content-type", "").startswith("image/")
    except requests.RequestException:
        return False

# Function to fetch one image URL for a given category
def fetch_image(category):
    try:
        ddg = DDGS()
        results = ddg.images(category, max_results=1)  # Retrieve only 1 image per category
        image_url = results[0]["image"] if results and validate_url(results[0]["image"]) else None
        return category, image_url
    except Exception as e:
        print(f"⚠️ Error fetching {category}: {e}")
        return category, None

# Function to download and display images safely
def display_image(category, image_url):
    if not image_url:
        print(f"⚠️ No valid image found for {category}")
        return

    try:
        response = requests.get(image_url, timeout=10)
        if "image" not in response.headers.get("content-type", ""):
            print(f"⚠️ Invalid image response for {category}: {image_url}")
            return

        img = Image.open(BytesIO(response.content))
        img.verify()  # Check for corruption before displaying
        img.show()
        print(f"✅ {category}: {image_url}")

    except Exception as e:
        print(f"❌ Error displaying image for {category}: {image_url}")
        print(traceback.format_exc())  # Show detailed error traceback

# Run parallel execution using ThreadPoolExecutor
with concurrent.futures.ThreadPoolExecutor() as executor:
    future_to_category = {executor.submit(fetch_image, category): category for category in categories}

    for future in concurrent.futures.as_completed(future_to_category):
        category, image_url = future.result()
        display_image(category, image_url)

print("✅ Completed parallel image mining and display!")


In [ ]:
import psycopg2
import urllib.request

def save_images(category, image_urls, cursor, conn):
    dataset_splits = ["train"] * 5 + ["val"] * 3 + ["test"] * 2

    for idx, url in enumerate(image_urls):
        folder_path = os.path.join(base_dir, dataset_splits[idx], category)
        os.makedirs(folder_path, exist_ok=True)
        image_path = os.path.join(folder_path, f"{category}_{idx}.jpg")

        try:
            # Download Image
            urllib.request.urlretrieve(url, image_path)
            print(f"✅ Saved {image_path}")

            # Insert metadata into PostgreSQL using functions
            cursor.execute("SELECT save_image_metadata(%s, %s, %s)", (url, image_path, "Unknown"))
            image_id = cursor.fetchone()[0]

            # Assign category & partition
            cursor.execute("SELECT save_image_category(%s, %s, %s)", (image_id, category, "low"))
            cursor.execute("SELECT save_dataset_partition(%s, %s, %s)", (image_id, dataset_splits[idx], "Web_scraped"))

            conn.commit()

        except Exception as e:
            print(f"❌ Failed to save {image_path}: {e}")


In [ ]:
# Initialize Selenium WebDriver correctly
service = Service(driver_path)
driver = webdriver.Chrome(service=service)

# Run scraping & save images
for category in categories:
    image_urls = scrape_images(category)
    if image_urls:
        save_images(category, image_urls, cursor, conn)
    else:
        print(f"⚠️ No images found for {category}, skipping save.")

# Close Selenium WebDriver
driver.quit()
print("✅ Completed image scraping and saving!")

# Close Database Connection
cursor.close()
conn.close()


In [ ]:
# Verify saved images
import os

for split in ["train", "val", "test"]:
    split_path = os.path.join(base_dir, split)
    if os.path.exists(split_path):
        for category in os.listdir(split_path):
            category_path = os.path.join(split_path, category)
            images = os.listdir(category_path) if os.path.isdir(category_path) else []
            print(f"📂 {category} ({split}): {len(images)} images")
    else:
        print(f"⚠️ {split} directory does not exist.")
